In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import os

from datetime import datetime, timezone
from uuid import uuid4

from dotenv import load_dotenv
from google.cloud import bigquery

from absolute_pipeline.comicvine import fetch_paginated_results

from absolute_pipeline.control import (get_control_state,detect_volume_changes,prepare_volume_control_updates, load_staging_records)

from absolute_pipeline.ingestion import (prepare_volume_records,load_volume_records, persist_volume_changes)

In [3]:
load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL ="https://comicvine.gamespot.com/api/"

client = bigquery.Client()

run_id = str(uuid4())
extracted_at = datetime.now(timezone.utc)

print("Project:", client.project)
print("Run ID:", run_id)
print("Extracted at:", extracted_at)

Project: dc-absolute
Run ID: 17253a37-327a-41f3-b096-d59eae780f8c
Extracted at: 2026-11-27 02:00:06.072427+00:00


In [4]:
all_results = fetch_paginated_results(
    base_url=BASE_URL,
    api_key=API_KEY,
    endpoint="volumes/",
    filters="name:Absolute"
)

print(f"Total volumes: {len(all_results)}")

Fetched 100 records. Accum Records: 100. Number of total results: 174
waiting...
Fetched 74 records. Accum Records: 174. Number of total results: 174
Final count: 174
Total volumes: 174


In [5]:
volume_ids = [volume["id"] for volume in all_results]

assert len(all_results) > 0
assert len(volume_ids) == len(set(volume_ids))

print(f"Total volumes: {len(all_results)}")

Total volumes: 174


In [6]:
control_state = get_control_state(
    client=client,
    table_id=f"{client.project}.control.volumes"
)

print(f"Control records: {len(control_state)}")

Control records: 174


In [7]:
control_state = get_control_state(
    client=client,
    table_id=f"{client.project}.control.volumes"
)

print(f"Control records: {len(control_state)}")

Control records: 174


In [8]:
changes = detect_volume_changes(
    volumes=all_results,
    control_state=control_state
)

for category, records in changes.items():
    print(f"{category}: {len(records)}")

new: 0
updated: 0
unchanged: 174


In [16]:
assert sum(len(records) for records in changes.values()) == len(all_results)

In [9]:
import pandas as pd

changed_volumes = changes["new"] + changes["updated"]

pd.json_normalize(changed_volumes)

""


In [10]:
loaded_rows = persist_volume_changes(
    client=client,
    changes=changes,
    table_id=f"{client.project}.bronze.volumes",
    run_id=run_id,
    extracted_at=extracted_at
)

print(f"Loaded rows: {loaded_rows}")

Loaded rows: 0


In [20]:
expected_rows = len(changes["new"]) + len(changes["updated"])

assert loaded_rows == expected_rows

In [11]:
import pandas as pd

raw_df = pd.json_normalize(all_results, sep="_")

raw_df["start_year"] = pd.to_numeric(
    raw_df["start_year"],
    errors="coerce"
)

filtered_df = raw_df[
    (raw_df["publisher_name"] == "DC Comics")
    & (raw_df["start_year"] >= 2024)
    & (~raw_df["name"].str.contains("Absolute Power", na=False))
].copy()

tracked_volume_ids = set(filtered_df["id"].astype(int))

In [15]:
control_records = prepare_volume_control_updates(
    changes=changes,
    control_state=control_state,
    tracked_volume_ids=tracked_volume_ids,
    run_id=run_id,
    extracted_at=extracted_at,
)

print(f"Prepared control records: {len(control_records)}")

Prepared control records: 174


In [16]:
import json 

assert len(control_records) == len(all_results)

assert len(control_records) == len({
    record["volume_id"]
    for record in control_records
})

assert all(
    record["run_id"] == run_id
    for record in control_records
)

json.dumps(control_records)

'[{"volume_id": 9336, "issue_count": 4, "last_issue_id": 74774, "source_updated_at": null, "last_checked_at": "2026-11-27T02:00:06.072427+00:00", "last_fetched_at": "2026-11-26T00:01:27.566074+00:00", "record_hash": "2eb64f76eaeec1db6209f1f3868dc06a80d48735a27bf74f2acb26ac63970d15", "is_tracked": false, "run_id": "17253a37-327a-41f3-b096-d59eae780f8c"}, {"volume_id": 18627, "issue_count": 1, "last_issue_id": 109878, "source_updated_at": null, "last_checked_at": "2026-11-27T02:00:06.072427+00:00", "last_fetched_at": "2026-11-26T00:01:27.566074+00:00", "record_hash": "a9fff44c82e039db7198904757384d20aef3fd5da463b9d1e55b209ce6280c51", "is_tracked": false, "run_id": "17253a37-327a-41f3-b096-d59eae780f8c"}, {"volume_id": 22759, "issue_count": 6, "last_issue_id": 136510, "source_updated_at": null, "last_checked_at": "2026-11-27T02:00:06.072427+00:00", "last_fetched_at": "2026-11-26T00:01:27.566074+00:00", "record_hash": "a95ed98e748722e7333e28bc4ee16877468ddcd42c5962f8d4206ceaa821d18d", "is_

In [20]:
df= pd.DataFrame(control_records)
len(df[df["is_tracked"] == True])

23

In [24]:
staging_table_id = f"{client.project}.control.volumes_staging"

staging_job = load_staging_records(
    client=client,
    records=control_records,
    table_id=staging_table_id,
)

8bdd1caf-118d-4a93-9026-f623a43f38af
DONE
174
None


In [25]:
assert staging_job.output_rows == len(control_records)

In [27]:
merge_query = """
MERGE control.volumes AS T
USING control.volumes_staging AS S
ON T.volume_id = S.volume_id

WHEN MATCHED THEN
    UPDATE SET
        T.issue_count = S.issue_count,
        T.last_issue_id = S.last_issue_id,
        T.source_updated_at = S.source_updated_at,
        T.last_checked_at = S.last_checked_at,
        T.last_fetched_at = S.last_fetched_at,
        T.record_hash = S.record_hash,
        T.is_tracked = S.is_tracked,
        T.run_id = S.run_id

WHEN NOT MATCHED THEN
    INSERT (
        volume_id,
        issue_count,
        last_issue_id,
        source_updated_at,
        last_checked_at,
        last_fetched_at,
        record_hash,
        is_tracked,
        run_id
    )
    VALUES (
        S.volume_id,
        S.issue_count,
        S.last_issue_id,
        S.source_updated_at,
        S.last_checked_at,
        S.last_fetched_at,
        S.record_hash,
        S.is_tracked,
        S.run_id
    );
"""

In [28]:
merge_job = client.query(merge_query)
merge_job.result()

print(f"Affected rows: {merge_job.num_dml_affected_rows}")

Affected rows: 174


In [29]:
updated_control_state = get_control_state(
    client=client,
    table_id=f"{client.project}.control.volumes"
)

In [30]:
for volume in changes["unchanged"]:
    volume_id = volume["id"]

    previous = control_state[volume_id]
    current = updated_control_state[volume_id]

    assert current["last_checked_at"] == extracted_at
    assert current["last_fetched_at"] == previous["last_fetched_at"]
    assert current["record_hash"] == previous["record_hash"]
    assert current["is_tracked"] == previous["is_tracked"]
    assert current["run_id"] == run_id

print("Control state validation passed")

Control state validation passed


In [31]:
from absolute_pipeline.control import update_volume_control_state

affected_rows = update_volume_control_state(
    client=client,
    changes=changes,
    control_state=control_state,
    tracked_volume_ids=tracked_volume_ids,
    control_table_id=f"{client.project}.control.volumes",
    staging_table_id=f"{client.project}.control.volumes_staging",
    run_id=run_id,
    extracted_at=extracted_at,
)

print(f"Control records synchronized: {affected_rows}")

Prepared control records: 174
2d786cf8-f7d4-4eb4-bfe7-e5d9bf5c83f9
DONE
174
None
Affected rows: 174
Control records synchronized: 174
